# B2-024-gpu-scientific-ml-capstone — Practice p10 — Solution

**Type:** constrained-coding · **Difficulty:** core · **Concepts:** scientific-ml-inverse-problems

*50 minutes.*  
**Set:** B  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261022`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C3-gradient-descent`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C10-competition-craft`, `book1:C11-neural-training`, `book1:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C3-gradient-descent](../../../../book1/units/C3-gradient-descent/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C10-competition-craft](../../../../book1/units/C10-competition-craft/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [book1:C12-classical-models](../../../../book1/units/C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb).

## Supplied cell (copied verbatim from the statement)

In [ ]:
import importlib.util
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261022
LOADER_CANDIDATES = (
    Path("../data/capstone_data.py"),
    Path("units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
    Path("book2/units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("capstone_data", loader_path)
capstone_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(capstone_data)
torch.set_num_threads(1)

SRC = torch.tensor([[0.3, 0.4], [0.75, 0.2]], dtype=torch.float64)
Q = torch.tensor([1.0, 1.5], dtype=torch.float64)
EXPECTED_FIELD = torch.tensor([
    [3.2000000000, 3.8095238095, 1.4035087719, 1.7777777778, 1.0958904110, 2.1621621622, 1.9512195122, 6.1538461538],
    [2.2556390977, 9.0909090909, 9.0909090909, 6.9767441860, 1.9607843137, 1.9607843137, 1.1857707510, 2.0979020979],
], dtype=torch.float64)
A_SMALL = torch.tensor([[1.0, 0.5, 0.0, 0.2], [0.0, 1.0, 0.5, 0.1], [0.3, 0.0, 1.0, 0.4]], dtype=torch.float64)
Y_SMALL = torch.tensor([1.0, 2.0, 0.5], dtype=torch.float64)
EXPECTED_TIK = torch.tensor([0.2320595001, 1.2237955911, 0.4371942912, 0.1186540996], dtype=torch.float64)
ATOL = 1e-9
RTOL = 1e-9

## Solution

**Forward field.** For sources `(N, 2)` and sensors `(S, 2)`, broadcasting `source_xy[:, None, :] - sensors_xy[None, :, :]` gives all `(N, S, 2)` offsets; summing their squares over the last axis gives $r^2$, and the field is $q/(r^2+h^2)$.

**Grid operator.** Column $j$ of $A$ is the sensor response to a *unit* source at grid node $j$, so $A$ is the transpose of the forward field of the 81 nodes with strength 1: $A=\mathcal F(\text{grid},\mathbf 1)^\top$, shape `(8, 81)`.

**Tikhonov solve.** The minimizer of $\lVert Au-y\rVert^2+\lambda\lVert u\rVert^2$ solves the normal equations $(A^\top A+\lambda I)u=A^\top y$ (practice p13 proves $A^\top A+\lambda I$ is invertible for every $\lambda>0$).
One `torch.linalg.solve` call handles both a vector right-hand side `(S,)` → `(G,)` and a batch of columns `(S, N)` → `(G, N)`; no inverse is ever formed.

In [ ]:
def forward_field(source_xy, strength, sensors_xy, height=capstone_data.SENSOR_HEIGHT):
    source_xy = source_xy.double()
    sensors_xy = sensors_xy.double()
    r2 = ((source_xy[:, None, :] - sensors_xy[None, :, :]) ** 2).sum(dim=-1)      # (N, S)
    return strength.double()[:, None] / (r2 + height ** 2)


def grid_operator(grid_xy, sensors_xy, height=capstone_data.SENSOR_HEIGHT):
    ones = torch.ones(grid_xy.shape[0], dtype=torch.float64)
    return forward_field(grid_xy, ones, sensors_xy, height).T                        # (S, G)


def tikhonov_solve(A, y, lam):
    A = A.double()
    y = y.double()
    M = A.T @ A + lam * torch.eye(A.shape[1], dtype=torch.float64)
    return torch.linalg.solve(M, A.T @ y)


def tikhonov_solve_no_lambda(A, y, lam):
    # Wrong variant: drops + lam * I
    A = A.double()
    return torch.linalg.solve(A.T @ A, A.T @ y.double())


# Probe 1
field = forward_field(SRC, Q, capstone_data.SENSORS_XY)
# Probe 2
A = grid_operator(capstone_data.GRID_XY, capstone_data.SENSORS_XY)
rank_AtA = int(torch.linalg.matrix_rank(A.T @ A))
# Probe 3
u_small = tikhonov_solve(A_SMALL, Y_SMALL, 0.5)
M_small = A_SMALL.T @ A_SMALL + 0.5 * torch.eye(4, dtype=torch.float64)
residual_small = torch.linalg.vector_norm(M_small @ u_small - A_SMALL.T @ Y_SMALL).item()
# Probe 4
val_x = capstone_data.val_rows("inverse")[0]
Y = val_x.double().T
U = tikhonov_solve(A, Y, capstone_data.TIKHONOV_LAMBDA)
u0 = tikhonov_solve(A, Y[:, 0], capstone_data.TIKHONOV_LAMBDA)
# Probe 5
density = U.T.clamp_min(0.0)                                   # (200, 81)
top = torch.topk(density, 3, dim=1)
mass = top.values.sum(dim=1, keepdim=True)
centroid = (top.values[..., None] * capstone_data.GRID_XY[top.indices]).sum(dim=1) / mass
estimates = torch.cat([centroid, mass], dim=1)
reference = capstone_data.tikhonov_grid_predict(val_x)
# Probe 6
try:
    u_wrong = tikhonov_solve_no_lambda(A_SMALL, Y_SMALL, 0.5)
    wrong_fails = not torch.allclose(u_wrong, EXPECTED_TIK, atol=ATOL, rtol=RTOL)
    wrong_outcome = f"returned {u_wrong.tolist()}"
except RuntimeError as err:
    wrong_fails = True
    wrong_outcome = "raised " + str(err).splitlines()[0][:100]
print("A shape", tuple(A.shape), "A[0,0]", A[0, 0].item(), "A[3,40]", A[3, 40].item(), "rank(A^T A) =", rank_AtA)
print("u_small", u_small.tolist(), "| residual", residual_small)
print("baseline max abs diff", (estimates.float() - reference).abs().max().item())
print("no-lambda variant:", wrong_outcome)

**Probe 6 — why $\lambda=0$ fails.**
$A^\top A$ is $81\times81$ but has rank at most 8 (it is a product through an 8-dimensional space; the probe measures exactly 8), so it is singular.
The normal equations $A^\top A u = A^\top y$ are nevertheless always consistent, because $A^\top y$ lies in the column space of $A^\top$, which equals the column space of $A^\top A$; a singular, consistent system has infinitely many solutions (any least-squares solution plus any vector in the null space of $A$, here at least $81-8=73$-dimensional).
Nothing in the data picks one of them, so a numerical solve of the singular matrix either raises or returns an essentially arbitrary vector, set by rounding error rather than by the data (often with huge entries).
(For the $4\times4$ probe, $A_{\text{small}}^\top A_{\text{small}}$ has rank 3 for the same reason.)

### Answer check

In [ ]:
# Probe 1
assert field.shape == (2, 8) and torch.allclose(field, EXPECTED_FIELD, atol=ATOL, rtol=RTOL)
# Probe 2
assert A.shape == (8, 81)
assert torch.isclose(A[0, 0], torch.tensor(16.0, dtype=torch.float64), atol=ATOL, rtol=RTOL)
assert torch.isclose(A[3, 40], torch.tensor(3.2, dtype=torch.float64), atol=ATOL, rtol=RTOL)
assert torch.allclose(A, forward_field(capstone_data.GRID_XY, torch.ones(81, dtype=torch.float64), capstone_data.SENSORS_XY).T,
                      atol=ATOL, rtol=RTOL)
assert rank_AtA == 8
# Probe 3
assert torch.allclose(u_small, EXPECTED_TIK, atol=ATOL, rtol=RTOL)
assert residual_small <= 1e-10
# Probe 4
assert U.shape == (81, 200)
assert torch.allclose(U[:, 0], u0, atol=ATOL, rtol=RTOL)
# Probe 5
assert estimates.shape == (200, 3)
assert torch.allclose(estimates.float(), reference, atol=1e-5, rtol=1e-5)
# Probe 6
assert wrong_fails